In [1]:
from langchain_community.document_loaders import PyPDFLoader

/var/folders/c2/rmzbv7kd1vj7ybrxfg00w2h00000gn/T/ipykernel_73003/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/pankajshah/Desktop/gen_ai/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "javanotes5.pdf"

In [3]:
loader = PyPDFLoader(file_path)
pages = loader.load()

len(pages)

699

In [4]:
pages[0].page_content[0:500]

'Introduction to Programming Using Java\nVersion 5.0, December 2006\n(Version 5.0.2, with minor corrections, November 2007)\nDavid J. Eck\nHobart and William Smith Colleges'

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators= ["\n\n","\n","."," "]
)

chunks = text_splitter.split_documents(pages)

print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content)

Number of chunks: 2279
Introduction to Programming Using Java
Version 5.0, December 2006
(Version 5.0.2, with minor corrections, November 2007)
David J. Eck
Hobart and William Smith Colleges


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6805.40it/s]


In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)


In [8]:
print(vectorstore._collection.count())

4558


In [9]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 3})

In [10]:
import getpass
import os

if "groq_open_api_key" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key:  ········


In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.8,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [12]:
def fetch_doc(question):
    docs = retriever.invoke(question)
    return "\n\n----\n\n".join(doc.page_content for doc in docs)

In [13]:
from langchain_core.prompts import ChatPromptTemplate

messages = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert Java instructor and mentor.

Your role is to teach Java programming concepts clearly and step-by-step.
Explain topics in a simple, beginner-friendly way with real-world examples.

Rules:
- Always explain like a teacher guiding a student
- Start from basics and gradually move to advanced concepts
- Provide Java code examples when needed
- Keep explanations clear, structured, and practical
- If a question is unclear, ask a clarifying question first

Context:
{context}

Answer:
"""
    ),
    ("human", "{question}")
])

In [14]:
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(lambda q: {
        "question": q,
        "context": fetch_doc(q)
    })
    | messages
    | llm
)

In [18]:
response = chain.invoke("What is JDK?")
print(response.content)

The **JDK (Java Development Kit)** is the essential toolkit for building, running, and managing Java programs. Let me explain it step by step in simple terms:

---

### **What is the JDK?**
Think of the JDK as your "toolkit" for creating and running Java programs. It includes:
1. **Java Compiler (`javac`)**: Converts your Java code (`MyProgram.java`) into machine-readable bytecode (`MyProgram.class`).
2. **Java Runtime Environment (JRE)**: Includes the Java Virtual Machine (JVM) to run Java programs.
3. **Development Tools**: Debuggers, documentation tools, and utilities for managing Java projects.
4. **Libraries**: Pre-written code (like `TextIO.java` in your book) that you can use in your programs.

---

### **Why is the JDK Needed?**
- If you **only** have the **JRE**, you can *run* Java apps (like games or appslets) but **cannot create** your own programs.
- The **JDK** is for **developers** who want to write, compile, and test Java code.

---

### **Key Components of the JDK**
1. 

In [15]:
response = chain.invoke("What is throws keyboard? and difference between throws and throw?")
print(response.content)

The **`throws` keyword** in Java is used in **method declarations** to indicate that a method may **throw** (not handle) certain exceptions to the caller. This is part of Java's exception handling mechanism. Let’s break it down with examples and clarify the difference between `throws` and `throw`.

---

### ✅ **`throw` vs `throws`: Key Differences**

| Feature          | `throw`                           | `throws`                          |
|-------------------|------------------------------------|------------------------------------|
| **Purpose**       | Used to **manually throw** an exception. | Used to **declare** exceptions a method might throw. |
| **Syntax**        | `throw new Exception();`         | `void method() throws Exception {}` |
| **Where Used**    | Inside method body (code logic).   | In method signature (declaration). |
| **Exception Type**| Throws **specific** exception.     | Declares **one or more** possible exceptions. |
| **Handling**      | Must be caught or 

In [16]:
response = chain.invoke("What is a literal?")
print(response.content)

A **literal** in Java is a fixed value written directly into the code, representing a specific data value. Literals are used to assign values to variables, constants, or in expressions. They are not variable names but direct representations of data. Let’s break this down with examples.

---

### **Types of Literals in Java**
Java supports several types of literals, each with its own syntax and purpose:

---

#### **1. Numeric Literals**
- **Integer Literals**: Represent whole numbers.
  - **Decimal (Base 10)**:  
    ```java
    int age = 25; // 25 is a decimal integer literal
    ```
  - **Octal (Base 8)**: Starts with a leading `0`.  
    ```java
    int octalValue = 045; // Equals 37 in decimal (4*8 + 5)
    ```
  - **Hexadecimal (Base 16)**: Starts with `0x` or `0X`.  
    ```java
    int hexValue = 0x1A; // Equals 26 in decimal (1*16 + 10)
    ```

- **Floating-Point Literals**: Represent decimal numbers.  
  ```java
  double price = 19.99; // 19.99 is a floating-point literal
  `